In [6]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigs
from scipy.stats import linregress
import mpmath
from numba import njit
import time

# ======== 1. 终极参数 ========
U_C = 1.543689012  
STEPS = 10**8      # 跑 1 亿步验证前 100 个零点
N_BINS = 10000     
N_ZEROS = 100      
N_OFFSET = float(10**20)  # 宇宙平滑启动时间补偿

# ======== 2. 召唤大自然的真实底牌 ========
mpmath.mp.dps = 15
true_zeros = np.array([float(mpmath.zetazero(i).imag) for i in range(1, N_ZEROS + 1)])

# ======== 3. Numba 极速演化 (究极高阶 k(n) 引擎) ========
@njit
def run_ultimate_universe(steps, n_bins, u_c, n_offset):
    transitions = np.zeros((n_bins, n_bins), dtype=np.uint32)
    x = 0.5
    last_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
    
    for i in range(1, steps + 1):
        # 真实的时间尺度 n
        n = i + n_offset
        
        # 【王博士的物理神迹：高阶微扰展开】
        # k_n 不再是常数 12.32，而是随着 n 极其缓慢衰减的函数！
        k_n = 4.7 + 10.13 / np.log(n)
        
        # 核心物理法则：包含 O(1/ln^2) 和 O(1/ln^3) 双重修正的退火过程
        mu = u_c + k_n / (np.log(n)**2)
        
        # 迭代更新
        x = 1.0 - mu * x * x
        
        # 极其平滑的参数曲线，几乎不会触发边界反弹
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
            
        current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
        transitions[last_bin, current_bin] += 1
        last_bin = current_bin
        
    return transitions

print(f"🚀 启动究极高阶微扰退火引擎 (N_OFFSET={N_OFFSET})...")
start_time = time.time()
trans_matrix = run_ultimate_universe(STEPS, N_BINS, U_C, N_OFFSET)

# ======== 4. 提取与解卷绕 ========
P_sparse = sp.csr_matrix(trans_matrix, dtype=np.float64)
row_sums = np.array(P_sparse.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1.0 
P_sparse.data /= row_sums[P_sparse.indices]

eigenvalues, _ = eigs(P_sparse, k=N_ZEROS + 50, which='LM', tol=1e-5)
phases = np.angle(eigenvalues)
valid_indices = (np.abs(eigenvalues.imag) > 1e-4) 
sorted_phases = np.sort(phases[valid_indices])
unwrapped_phases = np.unwrap(sorted_phases)[:N_ZEROS]

# ======== 5. 宏观对齐与预测 ========
slope, intercept, r_value, _, _ = linregress(unwrapped_phases, true_zeros)
predicted_zeros = slope * unwrapped_phases + intercept

# ======== 6. 极度硬核的对撞报表 ========
print(f"✅ 演化完成！耗时: {time.time() - start_time:.2f} 秒\n")
print("="*72)
print(f"   【究极引擎 (k_n = 4.7 + 10.13/ln(n))：前 100 零点对撞报表】")
print(f"   【模型 R² 相关系数: {r_value**2:.6f}】")
print("="*72)
print(f"{'Index (n)':<10} | {'预测位置 (Predicted)':<20} | {'真实位置 (True)':<15} | {'绝对误差 (Error)'}")
print("-" * 72)

for i in range(20):
    err = abs(predicted_zeros[i] - true_zeros[i])
    print(f"n = {i+1:<5} | {predicted_zeros[i]:<20.4f} | {true_zeros[i]:<15.4f} | {err:.4f}")

print("... (中间省略) ...")

for i in range(N_ZEROS - 5, N_ZEROS):
    err = abs(predicted_zeros[i] - true_zeros[i])
    print(f"n = {i+1:<5} | {predicted_zeros[i]:<20.4f} | {true_zeros[i]:<15.4f} | {err:.4f}")
print("="*72)
mean_err = np.mean(np.abs(predicted_zeros - true_zeros))
print(f"🎯 究极模型：前 {N_ZEROS} 个零点的平均绝对误差: {mean_err:.4f}")

🚀 启动究极高阶微扰退火引擎 (N_OFFSET=1e+20)...
✅ 演化完成！耗时: 6.08 秒

   【究极引擎 (k_n = 4.7 + 10.13/ln(n))：前 100 零点对撞报表】
   【模型 R² 相关系数: 0.992962】
Index (n)  | 预测位置 (Predicted)     | 真实位置 (True)     | 绝对误差 (Error)
------------------------------------------------------------------------
n = 1     | 28.8889              | 14.1347         | 14.7542
n = 2     | 30.8027              | 21.0220         | 9.7806
n = 3     | 36.8768              | 25.0109         | 11.8660
n = 4     | 38.7365              | 30.4249         | 8.3116
n = 5     | 40.2521              | 32.9351         | 7.3171
n = 6     | 45.6902              | 37.5862         | 8.1041
n = 7     | 50.1480              | 40.9187         | 9.2292
n = 8     | 50.5688              | 43.3271         | 7.2417
n = 9     | 51.6801              | 48.0052         | 3.6749
n = 10    | 53.2373              | 49.7738         | 3.4634
n = 11    | 53.6265              | 52.9703         | 0.6562
n = 12    | 58.5625              | 56.4462         | 2.1162
n = 13   